# Analyze difference between current headroom and projected electric grid load

This notebook analyzes the difference between the hosting capacity calculated utilizing current ICA data and projected electricity needs in California at the census tract level.

In [1]:
# load libraries
import pandas as pd 
import numpy as np
import geopandas as gpd

In [2]:
# load in data
load_projected = pd.read_csv('/../../capstone/electrigrid/data/projected_load/tract_scenario_comparison.csv')
tract_summary = pd.read_csv('/../../capstone/electrigrid/data/projected_load/tract_summary.csv')
sce_hostcap = gpd.read_parquet('/../../capstone/electrigrid/data/census_hosting_capacity/sce_census_map_new.parquet')

# Investigate the tract summary data

In [3]:
tract_summary

,tract_id,scenario,annual_heating,peak_heating,peak_hour_heating,annual_cooling,peak_cooling,peak_hour_cooling,annual_hvac,peak_hvac,...,monthly_average_peak_exports_m03,monthly_average_peak_exports_m04,monthly_average_peak_exports_m05,monthly_average_peak_exports_m06,monthly_average_peak_exports_m07,monthly_average_peak_exports_m08,monthly_average_peak_exports_m09,monthly_average_peak_exports_m10,monthly_average_peak_exports_m11,monthly_average_peak_exports_m12
0,101050101,baseline,2355214.704,2472.808,1303,6637557.742,5576.662,4936,9.049241e+06,5590.470,...,1450.209626,1833.681377,1925.371581,1700.288787,1331.944806,1490.284293,1598.494210,1495.433939,969.630944,856.646095
1,101050102,baseline,2270635.655,2225.642,1303,3836218.190,3429.123,4936,6.142196e+06,3437.887,...,565.807475,727.653351,776.119432,698.827671,563.063739,613.761342,654.739613,603.548389,383.965592,327.047146
2,101050201,baseline,1507129.516,1523.182,1278,2876298.696,2485.575,4936,4.408875e+06,2496.747,...,158.370717,198.168605,210.526255,188.158894,150.604208,165.391451,176.539408,161.680260,100.981237,88.425195
3,101050202,baseline,2591524.081,2324.490,1272,3525976.409,3128.553,4936,6.146741e+06,3139.377,...,131.507766,164.269276,173.048946,153.991553,122.185436,135.593279,144.068135,134.811911,85.319428,74.025202
4,101050301,baseline,1062507.516,1051.209,1277,2414380.377,2237.497,4936,3.496785e+06,2242.501,...,593.455756,741.788015,773.607302,673.806714,524.436341,590.561161,640.429973,603.156111,394.726014,356.693877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18079,99003906,high,1719231.308,2243.824,1302,4207000.506,3211.390,4936,5.964284e+06,3223.346,...,383.247966,665.251178,690.472134,523.604513,194.230573,300.051751,371.564583,387.353959,185.448970,60.907252
18080,99003907,high,2398806.229,3365.878,1303,9396687.363,7891.053,4936,1.185637e+07,7913.257,...,964.646567,1595.539565,1621.402697,1147.501466,450.938790,662.854387,799.109324,868.029848,488.027393,226.569533
18081,99003908,high,1068847.430,1314.966,1303,2001602.833,1646.402,4936,3.090531e+06,1654.150,...,200.699913,367.139869,396.191972,327.814842,155.414983,216.683595,231.372468,229.347478,87.068210,28.059093
18082,99003909,high,2071215.277,2595.438,1302,7018196.062,5675.536,4936,9.137991e+06,5691.564,...,613.824155,1026.484421,1031.436301,761.895102,351.567681,471.172587,564.965073,597.766374,327.811454,141.318573


The `tract_summary` is the data for the baseline. This won't be used further in this investigation. 

## Investigate Projected Load data

In [4]:
load_projected

,tract_id,annual_heating_baseline,peak_heating_baseline,peak_hour_heating_baseline,annual_cooling_baseline,peak_cooling_baseline,peak_hour_cooling_baseline,annual_hvac_baseline,peak_hvac_baseline,peak_hour_hvac_baseline,...,pct_increase_monthly_average_peak_exports_m03,pct_increase_monthly_average_peak_exports_m04,pct_increase_monthly_average_peak_exports_m05,pct_increase_monthly_average_peak_exports_m06,pct_increase_monthly_average_peak_exports_m07,pct_increase_monthly_average_peak_exports_m08,pct_increase_monthly_average_peak_exports_m09,pct_increase_monthly_average_peak_exports_m10,pct_increase_monthly_average_peak_exports_m11,pct_increase_monthly_average_peak_exports_m12
0,101050101,2355214.704,2472.808,1303,6637557.742,5576.662,4936,9.049241e+06,5590.470,4936,...,45.124032,59.776797,62.726066,54.604039,30.887606,38.191827,39.879113,38.829196,20.096738,9.059913
1,101050102,2270635.655,2225.642,1303,3836218.190,3429.123,4936,6.142196e+06,3437.887,4936,...,62.475768,82.018839,83.649028,69.599404,30.042981,45.787471,53.004921,55.495222,27.575736,9.307794
2,101050201,1507129.516,1523.182,1278,2876298.696,2485.575,4936,4.408875e+06,2496.747,4936,...,148.455989,191.512217,194.126877,167.827969,82.771258,116.739782,137.229268,140.855593,82.776129,50.921550
3,101050202,2591524.081,2324.490,1272,3525976.409,3128.553,4936,6.146741e+06,3139.377,4936,...,195.809506,250.741196,254.917695,211.899277,85.462427,138.467900,175.994677,185.914195,103.194909,57.780798
4,101050301,1062507.516,1051.209,1277,2414380.377,2237.497,4936,3.496785e+06,2242.501,4936,...,41.179062,53.806938,54.653630,44.065369,23.851800,30.238548,32.918991,33.853377,20.012565,13.345708
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9037,99003906,1567539.817,1675.641,1303,4007163.925,3009.648,4936,5.612756e+06,3021.604,4936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9038,99003907,1643612.243,1880.696,1303,9027033.800,7425.407,4936,1.073153e+07,7447.611,4936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9039,99003908,980829.110,1144.821,1303,1899860.020,1496.900,4936,2.900770e+06,1504.648,4936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9040,99003909,1688330.711,1672.609,1303,6871341.492,5477.952,4936,8.608252e+06,5493.980,4936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
load_projected.shape

(9042, 209)

There are 209 columns let's see which are the most important for us when looking at projected load data.

In [6]:
# subset for what was included in the readme
load_projected = load_projected[['tract_id', 
                                 'annual_pv_grid_baseline', 
                                 'peak_pv_grid_baseline',
                                 'annual_total_baseline',
                                 'peak_total_baseline',
                                 'annual_net_baseline',
                                 'peak_net_baseline',
                                 'pct_increase_annual_pv_grid',
                                 'p95_daily_peak_total_baseline',
                                 'p95_daily_peak_net_baseline',
                                 'p95_daily_peak_exports_baseline',
                                 'pct_increase_annual_total',
                                 'pct_increase_peak_total',
                                 'pct_increase_annual_net',
                                 'pct_increase_peak_net',
                                 'pct_increase_p95_daily_peak_total',
                                 'pct_increase_p95_daily_peak_net',
                                 'pct_increase_p95_daily_peak_exports']]

In [7]:
load_projected.head()

,tract_id,annual_pv_grid_baseline,peak_pv_grid_baseline,annual_total_baseline,peak_total_baseline,annual_net_baseline,peak_net_baseline,pct_increase_annual_pv_grid,p95_daily_peak_total_baseline,p95_daily_peak_net_baseline,p95_daily_peak_exports_baseline,pct_increase_annual_total,pct_increase_peak_total,pct_increase_annual_net,pct_increase_peak_net,pct_increase_p95_daily_peak_total,pct_increase_p95_daily_peak_net,pct_increase_p95_daily_peak_exports
0,101050101,2.959735e+06,2353.013571,2.418041e+07,7541.596242,2.165261e+07,6898.598399,35.885901,6785.780642,6051.076242,2103.024550,6.270672,5.083126,-7.017575,-7.658958,4.775366,-14.575055,69.942161
1,101050102,1.228744e+06,934.608793,1.478831e+07,4540.656719,1.388782e+07,4329.817895,44.122090,4047.703895,3758.999385,838.258763,6.343100,7.096314,-4.661723,-6.223291,6.411754,-9.571121,91.817151
2,101050201,3.304704e+05,252.035794,1.068719e+07,3870.539240,1.043091e+07,3814.111310,110.129063,2987.648443,2906.069323,228.615048,8.519428,4.290516,-1.464980,-2.510222,6.321410,-8.192671,216.261314
3,101050202,2.713197e+05,212.348648,1.334280e+07,4788.374193,1.313129e+07,4706.692714,133.994954,3675.509865,3600.023866,189.725772,9.048441,5.503752,-0.065857,-0.124648,8.068247,-3.404109,281.915351
4,101050301,1.202807e+06,945.061611,8.804029e+06,2933.921079,7.788774e+06,2702.590338,32.629535,2599.003287,2333.947411,850.930057,8.486157,8.183152,-4.197976,-0.988236,7.508347,-9.124383,61.359333


All of the projected load data is in percent. We need the actual values to compare them to the current headroom. 

## Calculate the projections of load at different intrevals


In [20]:
# calculate the yearly change based on the percent increase annual total
yearly_change = (load_projected['pct_increase_annual_total'] * 100) * load_projected['annual_total_baseline']

# project ten years from the data 
load_projected['ten_year_projection_annual_total'] = load_projected['annual_total_baseline'] + (yearly_change * 10)

# project twenty years from the data
load_projected['twenty_year_projection_annual_total'] = load_projected['annual_total_baseline'] + (yearly_change * 30)

/tmp/ipykernel_3687841/1596595634.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  load_projected['ten_year_projection_annual_total'] = load_projected['annual_total_baseline'] + (yearly_change * 10)
/tmp/ipykernel_3687841/1596595634.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  load_projected['twenty_year_projection_annual_total'] = load_projected['annual_total_baseline'] + (yearly_change * 30)


In [21]:
load_projected

,tract_id,annual_pv_grid_baseline,peak_pv_grid_baseline,annual_total_baseline,peak_total_baseline,annual_net_baseline,peak_net_baseline,pct_increase_annual_pv_grid,p95_daily_peak_total_baseline,p95_daily_peak_net_baseline,...,pct_increase_peak_net,pct_increase_p95_daily_peak_total,pct_increase_p95_daily_peak_net,pct_increase_p95_daily_peak_exports,ten_year_projection,twenty_year_projection,ten_year_projection_annual_total,twenty_year_projection_annual_total,ten_year_projection_annual_net,twenty_year_projection_annual_net
0,101050101,2.959735e+06,2353.013571,2.418041e+07,7541.596242,2.165261e+07,6898.598399,35.885901,6785.780642,6051.076242,...,-7.658958,4.775366,-14.575055,69.942161,1.540455e+09,4.573003e+09,1.516516e+11,4.549065e+11,-1.519271e+11,-4.558247e+11
1,101050102,1.228744e+06,934.608793,1.478831e+07,4540.656719,1.388782e+07,4329.817895,44.122090,4047.703895,3758.999385,...,-6.223291,6.411754,-9.571121,91.817151,9.528256e+08,2.828900e+09,9.381852e+10,2.814260e+11,-6.472728e+10,-1.942096e+11
2,101050201,3.304704e+05,252.035794,1.068719e+07,3870.539240,1.043091e+07,3814.111310,110.129063,2987.648443,2906.069323,...,-2.510222,6.321410,-8.192671,216.261314,9.211745e+08,2.742149e+09,9.105942e+10,2.731569e+11,-1.527064e+10,-4.583278e+10
3,101050202,2.713197e+05,212.348648,1.334280e+07,4788.374193,1.313129e+07,4706.692714,133.994954,3675.509865,3600.023866,...,-0.124648,8.068247,-3.404109,281.915351,1.220658e+09,3.635288e+09,1.207449e+11,3.622079e+11,-8.516563e+08,-2.581231e+09
4,101050301,1.202807e+06,945.061611,8.804029e+06,2933.921079,7.788774e+06,2702.590338,32.629535,2599.003287,2333.947411,...,-0.988236,7.508347,-9.124383,61.359333,7.559278e+08,2.250175e+09,7.472118e+10,2.241459e+11,-3.268930e+10,-9.808348e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9037,99003906,0.000000e+00,0.000000,1.455558e+07,4185.300927,1.455558e+07,4185.300927,NaN,3884.556449,3884.556449,...,-12.288703,5.914428,-12.992745,NaN,9.126978e+08,2.708982e+09,8.982878e+10,2.694572e+11,-8.044292e+10,-2.413579e+11
9038,99003907,0.000000e+00,0.000000,2.625728e+07,9380.366246,2.625728e+07,9380.366246,NaN,8510.679272,8510.679272,...,-10.933349,5.819691,-13.728715,NaN,2.481326e+09,7.391463e+09,2.455331e+11,7.365468e+11,-1.884842e+11,-5.655051e+11
9039,99003908,0.000000e+00,0.000000,7.649701e+06,2141.476694,7.649701e+06,2141.476694,NaN,1965.310851,1965.310851,...,-6.686299,8.798270,-9.864030,NaN,4.919656e+08,1.460597e+09,4.843924e+10,1.453024e+11,-3.938707e+10,-1.181765e+11
9040,99003909,0.000000e+00,0.000000,2.068289e+07,7127.533634,2.068289e+07,7127.533634,NaN,6449.004908,6449.004908,...,-13.528691,3.646313,-13.661151,NaN,1.369100e+09,4.065935e+09,1.348624e+11,4.045458e+11,-1.323050e+11,-3.969564e+11


In [15]:
# calculate the yearly change based on the percent increase annual net
yearly_change = (load_projected['pct_increase_annual_net'] * 100) * load_projected['annual_net_baseline']

# project ten years from the data 
load_projected['ten_year_projection_annual_net'] = load_projected['annual_net_baseline'] + (yearly_change * 10)

# project twenty years from the data
load_projected['twenty_year_projection_annual_net'] = load_projected['annual_net_baseline'] + (yearly_change * 30)

/tmp/ipykernel_3687841/593859276.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  load_projected['ten_year_projection_annual_net'] = load_projected['annual_net_baseline'] + (yearly_change * 10)
/tmp/ipykernel_3687841/593859276.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  load_projected['twenty_year_projection_annual_net'] = load_projected['annual_net_baseline'] + (yearly_change * 30)


In [16]:
load_projected

,tract_id,annual_pv_grid_baseline,peak_pv_grid_baseline,annual_total_baseline,peak_total_baseline,annual_net_baseline,peak_net_baseline,pct_increase_annual_pv_grid,p95_daily_peak_total_baseline,p95_daily_peak_net_baseline,...,pct_increase_peak_net,pct_increase_p95_daily_peak_total,pct_increase_p95_daily_peak_net,pct_increase_p95_daily_peak_exports,ten_year_projection,twenty_year_projection,ten_year_projection_annual_total,twenty_year_projection_annual_total,ten_year_projection_annual_net,twenty_year_projection_annual_net
0,101050101,2.959735e+06,2353.013571,2.418041e+07,7541.596242,2.165261e+07,6898.598399,35.885901,6785.780642,6051.076242,...,-7.658958,4.775366,-14.575055,69.942161,1.540455e+09,4.573003e+09,1.516516e+11,4.549065e+11,-1.519271e+11,-4.558247e+11
1,101050102,1.228744e+06,934.608793,1.478831e+07,4540.656719,1.388782e+07,4329.817895,44.122090,4047.703895,3758.999385,...,-6.223291,6.411754,-9.571121,91.817151,9.528256e+08,2.828900e+09,9.381852e+10,2.814260e+11,-6.472728e+10,-1.942096e+11
2,101050201,3.304704e+05,252.035794,1.068719e+07,3870.539240,1.043091e+07,3814.111310,110.129063,2987.648443,2906.069323,...,-2.510222,6.321410,-8.192671,216.261314,9.211745e+08,2.742149e+09,9.105942e+10,2.731569e+11,-1.527064e+10,-4.583278e+10
3,101050202,2.713197e+05,212.348648,1.334280e+07,4788.374193,1.313129e+07,4706.692714,133.994954,3675.509865,3600.023866,...,-0.124648,8.068247,-3.404109,281.915351,1.220658e+09,3.635288e+09,1.207449e+11,3.622079e+11,-8.516563e+08,-2.581231e+09
4,101050301,1.202807e+06,945.061611,8.804029e+06,2933.921079,7.788774e+06,2702.590338,32.629535,2599.003287,2333.947411,...,-0.988236,7.508347,-9.124383,61.359333,7.559278e+08,2.250175e+09,7.472118e+10,2.241459e+11,-3.268930e+10,-9.808348e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9037,99003906,0.000000e+00,0.000000,1.455558e+07,4185.300927,1.455558e+07,4185.300927,NaN,3884.556449,3884.556449,...,-12.288703,5.914428,-12.992745,NaN,9.126978e+08,2.708982e+09,8.982878e+10,2.694572e+11,-8.044292e+10,-2.413579e+11
9038,99003907,0.000000e+00,0.000000,2.625728e+07,9380.366246,2.625728e+07,9380.366246,NaN,8510.679272,8510.679272,...,-10.933349,5.819691,-13.728715,NaN,2.481326e+09,7.391463e+09,2.455331e+11,7.365468e+11,-1.884842e+11,-5.655051e+11
9039,99003908,0.000000e+00,0.000000,7.649701e+06,2141.476694,7.649701e+06,2141.476694,NaN,1965.310851,1965.310851,...,-6.686299,8.798270,-9.864030,NaN,4.919656e+08,1.460597e+09,4.843924e+10,1.453024e+11,-3.938707e+10,-1.181765e+11
9040,99003909,0.000000e+00,0.000000,2.068289e+07,7127.533634,2.068289e+07,7127.533634,NaN,6449.004908,6449.004908,...,-13.528691,3.646313,-13.661151,NaN,1.369100e+09,4.065935e+09,1.348624e+11,4.045458e+11,-1.323050e+11,-3.969564e+11


## View hosting capacity calculations

In [6]:
sce_hostcap.head()

,GEOID,zillow_tract_hh_count,avg_DER_remain_pv_hh,avg_DER_remain_load_hh,avg_DER_total_pv_hh,geometry
0,06065042516,776.0,6.02214,2.715849,7.550272,"POLYGON ((254831.161 -450428.536, 254879.224 -..."
1,06065042716,2174.0,0.00000,0.099297,4.067808,"POLYGON ((252988.226 -477491.812, 253070.244 -..."
2,06065042717,1781.0,NaN,NaN,NaN,"POLYGON ((253204.987 -475305.031, 253228.177 -..."
3,06065042902,1284.0,NaN,NaN,NaN,"POLYGON ((245709.226 -466855.471, 245710.230 -..."
4,06065042903,1417.0,NaN,NaN,NaN,"POLYGON ((244083.685 -461694.249, 244084.463 -..."
